# ZNCC alignment-QC validation v2

This read-only notebook compares two exact `full_image` aliases using dense, pre-warp local zero-normalized cross-correlation (ZNCC). It shows matched whole-slide and micron-coordinate zoom views, then aggregates the dense ZNCC measurements around the detected cell centers from `agg_cell_labels` and plots those cell values spatially.

The notebook does not modify the SpatialData store or run an upstream pipeline stage. Optional PNG export writes only the requested figures.

## Environment and configuration

Run this in the SpatialData environment with the alignment extra installed: `pip install -e '.[alignment-qc]'`. Set two exact existing aliases below. The alias text is not interpreted.

In [ ]:
from pathlib import Path
import math

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from spatialdata import read_zarr

from mif_pipeline import load_config
from mif_pipeline.config import get_slide_config, resolve_channel_entries
from mif_pipeline.alignment_qc import (
    _cell_observations,
    _channel_names,
    _image_levels,
    _materialize_channel,
    neighborhood_radii_pixels,
    normalize_percentile_image,
    sample_neighborhood_nanmedian,
    select_pyramid_level,
)

plt.rcParams['figure.dpi'] = 120

In [ ]:
CONFIG_PATH = Path('../example.yaml')
SLIDE_ID = 'SLIDE-0272'
REFERENCE_ALIAS = 'R1_DAPI'
MOVING_ALIAS = 'R2_DAPI'

TARGET_RESOLUTION_UM = 2.6
PYRAMID_LEVEL = None
LOWER_PERCENTILE = 1.0
UPPER_PERCENTILE = 99.9

# Dense ZNCC settings. Window size is converted independently along x and y.
ZNCC_WINDOW_SIZE_UM = 75.0
ZNCC_MIN_VALID_FRACTION = 0.9
ZNCC_MIN_LOCAL_STD = 0.005
ZNCC_COMPUTE_CHUNKS = (1024, 1024)

# Dense values are aggregated around each cell center over this physical radius.
CELL_SAMPLING_RADIUS_UM = 2.6
CELL_MARKER_SIZE = 3

# Micron-coordinate zoom. None repeats the complete selected level.
# Example: ZOOM_X_UM = (8_000, 10_000); ZOOM_Y_UM = (12_000, 14_000)
ZOOM_X_UM = None
ZOOM_Y_UM = None

# Full-slide plots are display-decimated only; all calculations use every selected-level pixel.
MAX_PLOT_DIMENSION = 2500
FIGURE_DPI = 300
PLOT_OUTPUT_DIR = None  # Example: Path('zncc_validation_plots')

In [ ]:
config = load_config(CONFIG_PATH)
slide = get_slide_config(config, SLIDE_ID)
resolve_channel_entries(config, SLIDE_ID, [REFERENCE_ALIAS, MOVING_ALIAS])

store_path = Path(slide['spatialdata']['store_path'])
if not store_path.exists():
    raise FileNotFoundError(f'Canonical SpatialData store not found: {store_path}')
sdata = read_zarr(store_path)
if 'full_image' not in sdata.images:
    raise KeyError("Canonical store is missing images['full_image']")
if 'agg_cell_labels' not in sdata.tables:
    raise KeyError("Canonical store is missing tables['agg_cell_labels']")

levels = _image_levels(sdata.images['full_image'])
available_aliases = _channel_names(levels[0][1])
missing = [a for a in (REFERENCE_ALIAS, MOVING_ALIAS) if a not in available_aliases]
if missing:
    raise KeyError(f'Aliases absent from full_image: {missing}')

selected = select_pyramid_level(
    levels,
    native_pixel_size_um=float(slide['pixel_size_um']),
    pyramid_level=PYRAMID_LEVEL,
    target_resolution_um=TARGET_RESOLUTION_UM,
)
level_array = selected['array']
pixel_size_x_um = float(selected['pixel_size_x_um'])
pixel_size_y_um = float(selected['pixel_size_y_um'])

reference_raw = _materialize_channel(level_array, REFERENCE_ALIAS)
moving_raw = _materialize_channel(level_array, MOVING_ALIAS)
height, width = reference_raw.shape
reference_normalized, reference_bounds = normalize_percentile_image(
    reference_raw, lower_percentile=LOWER_PERCENTILE, upper_percentile=UPPER_PERCENTILE
)
moving_normalized, moving_bounds = normalize_percentile_image(
    moving_raw, lower_percentile=LOWER_PERCENTILE, upper_percentile=UPPER_PERCENTILE
)

print(f'Store: {store_path}')
print(f"Level: {selected['name']} | shape={reference_raw.shape}")
print(f'Resolution: {pixel_size_x_um:.4f} × {pixel_size_y_um:.4f} µm/pixel')
print(f'Reference bounds: {reference_bounds}')
print(f'Moving bounds: {moving_bounds}')

## Calculate dense local ZNCC

For every selected-level pixel, ZNCC is the Pearson correlation between corresponding pixels in overlapping reference and moving neighborhoods. The residual is `1 - clip(correlation, 0, 1)`: zero is strong agreement and one is uncorrelated or negatively correlated. Low-variance and insufficiently valid neighborhoods are `NaN`. OpenCV box filters calculate the required local sums in chunks with window-radius halos.

In [ ]:
def physical_odd_window(size_um, pixel_size_um):
    radius = max(1, int(np.ceil((float(size_um) / 2) / float(pixel_size_um))))
    return 2 * radius + 1


def dense_local_zncc(
    reference,
    comparison,
    *,
    window_shape,
    minimum_valid_fraction=0.9,
    minimum_local_std=0.005,
    chunk_shape=(1024, 1024),
):
    reference = np.asarray(reference, dtype=np.float32)
    comparison = np.asarray(comparison, dtype=np.float32)
    if reference.ndim != 2 or reference.shape != comparison.shape:
        raise ValueError('ZNCC inputs must be matching two-dimensional arrays.')

    window_y, window_x = map(int, window_shape)
    if window_y < 3 or window_x < 3 or window_y % 2 != 1 or window_x % 2 != 1:
        raise ValueError('ZNCC window dimensions must be odd integers >= 3.')
    radius_y, radius_x = window_y // 2, window_x // 2
    window_area = float(window_y * window_x)
    correlation_output = np.full(reference.shape, np.nan, dtype=np.float32)

    def box_sum(values):
        return cv2.boxFilter(
            values, ddepth=cv2.CV_64F, ksize=(window_x, window_y),
            normalize=False, borderType=cv2.BORDER_CONSTANT,
        )

    height, width = reference.shape
    chunk_y, chunk_x = map(int, chunk_shape)
    for y0 in range(0, height, chunk_y):
        y1 = min(y0 + chunk_y, height)
        ey0, ey1 = max(0, y0 - radius_y), min(height, y1 + radius_y)
        for x0 in range(0, width, chunk_x):
            x1 = min(x0 + chunk_x, width)
            ex0, ex1 = max(0, x0 - radius_x), min(width, x1 + radius_x)
            ref = reference[ey0:ey1, ex0:ex1]
            mov = comparison[ey0:ey1, ex0:ex1]
            valid = np.isfinite(ref) & np.isfinite(mov)
            weights = valid.astype(np.float32)
            ref = np.where(valid, ref, 0).astype(np.float32, copy=False)
            mov = np.where(valid, mov, 0).astype(np.float32, copy=False)

            count = box_sum(weights)
            safe_count = np.maximum(count, 1.0)
            mean_ref = box_sum(ref) / safe_count
            mean_mov = box_sum(mov) / safe_count
            var_ref = box_sum(ref * ref) / safe_count - mean_ref * mean_ref
            var_mov = box_sum(mov * mov) / safe_count - mean_mov * mean_mov
            covariance = box_sum(ref * mov) / safe_count - mean_ref * mean_mov
            np.maximum(var_ref, 0, out=var_ref)
            np.maximum(var_mov, 0, out=var_mov)
            std_ref = np.sqrt(var_ref)
            std_mov = np.sqrt(var_mov)
            denominator = std_ref * std_mov
            supported = (
                (count >= minimum_valid_fraction * window_area)
                & (std_ref >= minimum_local_std)
                & (std_mov >= minimum_local_std)
                & (denominator > 0)
            )
            correlation = np.zeros_like(covariance)
            np.divide(covariance, denominator, out=correlation, where=supported)
            correlation = np.clip(correlation, -1.0, 1.0)
            correlation[~supported] = np.nan

            cy0, cy1 = y0 - ey0, y1 - ey0
            cx0, cx1 = x0 - ex0, x1 - ex0
            correlation_output[y0:y1, x0:x1] = correlation[cy0:cy1, cx0:cx1].astype(np.float32)

    residual_output = 1.0 - np.clip(correlation_output, 0.0, 1.0)
    residual_output[~np.isfinite(correlation_output)] = np.nan
    return {
        'correlation': correlation_output,
        'residual': residual_output.astype(np.float32),
        'valid_mask': np.isfinite(correlation_output),
    }


def magenta_green_overlay(reference, comparison):
    reference = np.clip(np.asarray(reference, dtype=np.float32), 0, 1)
    comparison = np.clip(np.asarray(comparison, dtype=np.float32), 0, 1)
    return np.stack([reference, comparison, reference], axis=-1)


def micron_crop(image_shape, x_range_um=None, y_range_um=None):
    height, width = image_shape
    x_start, x_stop = (0.0, width * pixel_size_x_um) if x_range_um is None else map(float, x_range_um)
    y_start, y_stop = (0.0, height * pixel_size_y_um) if y_range_um is None else map(float, y_range_um)
    if x_stop <= x_start or y_stop <= y_start:
        raise ValueError('Micron coordinate ranges must be increasing.')
    x0 = max(0, int(np.floor(x_start / pixel_size_x_um)))
    x1 = min(width, int(np.ceil(x_stop / pixel_size_x_um)))
    y0 = max(0, int(np.floor(y_start / pixel_size_y_um)))
    y1 = min(height, int(np.ceil(y_stop / pixel_size_y_um)))
    if x1 <= x0 or y1 <= y0:
        raise ValueError('Coordinates do not overlap the selected pyramid level.')
    extent = [x0 * pixel_size_x_um, x1 * pixel_size_x_um, y1 * pixel_size_y_um, y0 * pixel_size_y_um]
    return np.s_[y0:y1, x0:x1], extent, (x0, x1, y0, y1)

In [ ]:
zncc_window_y = physical_odd_window(ZNCC_WINDOW_SIZE_UM, pixel_size_y_um)
zncc_window_x = physical_odd_window(ZNCC_WINDOW_SIZE_UM, pixel_size_x_um)
zncc = dense_local_zncc(
    reference_normalized,
    moving_normalized,
    window_shape=(zncc_window_y, zncc_window_x),
    minimum_valid_fraction=ZNCC_MIN_VALID_FRACTION,
    minimum_local_std=ZNCC_MIN_LOCAL_STD,
    chunk_shape=ZNCC_COMPUTE_CHUNKS,
)
zncc_correlation = zncc['correlation']
zncc_residual = zncc['residual']
zncc_valid = zncc['valid_mask']

print(
    f'ZNCC window: {zncc_window_x} × {zncc_window_y} pixels '
    f'({zncc_window_x * pixel_size_x_um:.1f} × {zncc_window_y * pixel_size_y_um:.1f} µm)'
)
print(f'Valid dense fraction: {np.mean(zncc_valid):.3f}')
print(f'Median valid correlation: {np.nanmedian(zncc_correlation):.3f}')
print(f'Median valid residual: {np.nanmedian(zncc_residual):.3f}')

## Whole-slide comparison

All image and ZNCC panels share the same physical extent. Reference-only signal is magenta, moving-only signal is green, and agreement in the overlay is white. Full-slide arrays are decimated only for display. Correlation and residual color limits are fixed, making those colors comparable with the zoom figure.

In [ ]:
plot_step = max(1, math.ceil(max(reference_normalized.shape) / MAX_PLOT_DIMENSION))
plot_slice = np.s_[::plot_step, ::plot_step]
height, width = reference_normalized.shape
full_extent_um = [0, width * pixel_size_x_um, height * pixel_size_y_um, 0]
reference_plot = reference_normalized[plot_slice]
moving_plot = moving_normalized[plot_slice]
overlay_plot = magenta_green_overlay(reference_plot, moving_plot)
hist_values = zncc_residual[plot_slice]
hist_values = hist_values[np.isfinite(hist_values)]

whole_fig, axes = plt.subplots(2, 3, figsize=(20, 13), constrained_layout=True)
image_panels = [
    (reference_plot, f'Reference: {REFERENCE_ALIAS}', 'gray', 0, 1),
    (moving_plot, f'Moving: {MOVING_ALIAS}', 'gray', 0, 1),
    (overlay_plot, 'Reference magenta / moving green', None, None, None),
    (zncc_correlation[plot_slice], 'Local ZNCC correlation', 'coolwarm', -1, 1),
    (zncc_residual[plot_slice], 'Local ZNCC residual', 'inferno', 0, 1),
]
for ax, (image, title, cmap, vmin, vmax) in zip(axes.flat[:5], image_panels):
    if image.ndim == 3:
        shown = ax.imshow(image, extent=full_extent_um, origin='upper')
    else:
        shown = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, extent=full_extent_um, origin='upper')
        whole_fig.colorbar(shown, ax=ax, shrink=0.75)
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
axes.flat[5].hist(hist_values, bins=100, range=(0, 1), color='steelblue', alpha=0.9)
axes.flat[5].set_title('Display-sampled ZNCC residual profile')
axes.flat[5].set_xlabel('ZNCC residual')
axes.flat[5].set_ylabel('selected-level pixels')
whole_fig.suptitle(f'Whole-slide pre-warp ZNCC | {REFERENCE_ALIAS} vs {MOVING_ALIAS}', fontsize=16)

if PLOT_OUTPUT_DIR is not None:
    output_dir = Path(PLOT_OUTPUT_DIR).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)
    whole_path = output_dir / f'{SLIDE_ID}_zncc_whole.png'
    whole_fig.savefig(whole_path, dpi=FIGURE_DPI, bbox_inches='tight')
    print(f'Saved: {whole_path}')
plt.show()

## Matched micron-coordinate zoom

Set `ZOOM_X_UM` and `ZOOM_Y_UM` in the configuration cell, rerun from the configuration cell, and use this figure to compare the full selected-level overlay directly with the ZNCC map over exactly the same coordinates.

In [ ]:
zoom_slice, zoom_extent_um, zoom_pixels = micron_crop(reference_normalized.shape, ZOOM_X_UM, ZOOM_Y_UM)
reference_zoom = reference_normalized[zoom_slice]
moving_zoom = moving_normalized[zoom_slice]
overlay_zoom = magenta_green_overlay(reference_zoom, moving_zoom)
zoom_hist = zncc_residual[zoom_slice]
zoom_hist = zoom_hist[np.isfinite(zoom_hist)]

zoom_fig, axes = plt.subplots(2, 3, figsize=(20, 13), constrained_layout=True)
zoom_panels = [
    (reference_zoom, f'Reference: {REFERENCE_ALIAS}', 'gray', 0, 1),
    (moving_zoom, f'Moving: {MOVING_ALIAS}', 'gray', 0, 1),
    (overlay_zoom, 'Reference magenta / moving green', None, None, None),
    (zncc_correlation[zoom_slice], 'Local ZNCC correlation', 'coolwarm', -1, 1),
    (zncc_residual[zoom_slice], 'Local ZNCC residual', 'inferno', 0, 1),
]
for ax, (image, title, cmap, vmin, vmax) in zip(axes.flat[:5], zoom_panels):
    if image.ndim == 3:
        shown = ax.imshow(image, extent=zoom_extent_um, origin='upper')
    else:
        shown = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, extent=zoom_extent_um, origin='upper')
        zoom_fig.colorbar(shown, ax=ax, shrink=0.75)
    ax.set_title(title)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
axes.flat[5].hist(zoom_hist, bins=100, range=(0, 1), color='steelblue', alpha=0.9)
axes.flat[5].set_title('Zoom ZNCC residual profile')
axes.flat[5].set_xlabel('ZNCC residual')
axes.flat[5].set_ylabel('selected-level pixels')
zoom_fig.suptitle(
    f'Matched zoom | x={zoom_extent_um[0]:.1f}–{zoom_extent_um[1]:.1f} µm, '
    f'y={zoom_extent_um[3]:.1f}–{zoom_extent_um[2]:.1f} µm', fontsize=16
)

if PLOT_OUTPUT_DIR is not None:
    output_dir = Path(PLOT_OUTPUT_DIR).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)
    zoom_path = output_dir / f'{SLIDE_ID}_zncc_zoom.png'
    zoom_fig.savefig(zoom_path, dpi=FIGURE_DPI, bbox_inches='tight')
    print(f'Saved: {zoom_path}')
plt.show()

## Aggregate ZNCC at detected cell centers

Cell IDs and micron-space centers come directly from `agg_cell_labels`. Centers are projected onto the selected ZNCC grid, and each cell receives the neighborhood `nanmedian`. This is a regional alignment value centered on the cell, not an intensity measurement over the complete cell mask.

In [ ]:
source_obs, instance_ids, spatial_um = _cell_observations(sdata.tables['agg_cell_labels'])
x_level = spatial_um[:, 0] / pixel_size_x_um
y_level = spatial_um[:, 1] / pixel_size_y_um
radius_x, radius_y = neighborhood_radii_pixels(
    CELL_SAMPLING_RADIUS_UM,
    pixel_size_x_um=pixel_size_x_um,
    pixel_size_y_um=pixel_size_y_um,
)
cell_correlation = sample_neighborhood_nanmedian(
    zncc_correlation, x_level, y_level, radius_x=radius_x, radius_y=radius_y
)
cell_residual = sample_neighborhood_nanmedian(
    zncc_residual, x_level, y_level, radius_x=radius_x, radius_y=radius_y
)
nearest_x = np.clip(np.rint(x_level).astype(int), 0, width - 1)
nearest_y = np.clip(np.rint(y_level).astype(int), 0, height - 1)
center_residual = zncc_residual[nearest_y, nearest_x]

cell_qc = pd.DataFrame(
    {
        'instance_id': instance_ids,
        'x_um': spatial_um[:, 0],
        'y_um': spatial_um[:, 1],
        'zncc_correlation': cell_correlation,
        'zncc_residual': cell_residual,
        'center_pixel_residual': center_residual,
        'aggregation_delta': cell_residual - center_residual,
    }
).set_index('instance_id')

print(f'Cells: {len(cell_qc):,}')
print(f'Cell sampling window: {2 * radius_x + 1} × {2 * radius_y + 1} pixels')
print(f'Cells with valid aggregated ZNCC: {np.mean(np.isfinite(cell_residual)):.3f}')
cell_qc.describe(percentiles=[0.05, 0.5, 0.95]).T

In [ ]:
cell_view_slice, cell_extent_um, _ = micron_crop(reference_normalized.shape, ZOOM_X_UM, ZOOM_Y_UM)
x_left, x_right = cell_extent_um[0], cell_extent_um[1]
y_top, y_bottom = cell_extent_um[3], cell_extent_um[2]
in_view = (
    (cell_qc['x_um'] >= x_left) & (cell_qc['x_um'] <= x_right)
    & (cell_qc['y_um'] >= y_top) & (cell_qc['y_um'] <= y_bottom)
)
shown_cells = cell_qc.loc[in_view]

cell_fig, axes = plt.subplots(1, 3, figsize=(21, 7), constrained_layout=True)
dense_shown = axes[0].imshow(
    zncc_residual[cell_view_slice], cmap='inferno', vmin=0, vmax=1,
    extent=cell_extent_um, origin='upper',
)
axes[0].set_title('Dense ZNCC residual')
cell_fig.colorbar(dense_shown, ax=axes[0], shrink=0.75)

for ax, name, title, cmap, vmin, vmax in [
    (axes[1], 'zncc_residual', 'Aggregated cell ZNCC residual', 'inferno', 0, 1),
    (axes[2], 'zncc_correlation', 'Aggregated cell ZNCC correlation', 'coolwarm', -1, 1),
]:
    ax.imshow(
        reference_normalized[cell_view_slice], cmap='gray',
        extent=cell_extent_um, origin='upper', alpha=0.4,
    )
    points = ax.scatter(
        shown_cells['x_um'], shown_cells['y_um'], c=shown_cells[name],
        s=CELL_MARKER_SIZE, cmap=cmap, vmin=vmin, vmax=vmax, linewidths=0,
    )
    ax.set_title(title)
    cell_fig.colorbar(points, ax=ax, shrink=0.75)
for ax in axes:
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_aspect('equal')
cell_fig.suptitle(f'ZNCC cell aggregation | {len(shown_cells):,} cells in view', fontsize=16)

if PLOT_OUTPUT_DIR is not None:
    output_dir = Path(PLOT_OUTPUT_DIR).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)
    cell_path = output_dir / f'{SLIDE_ID}_zncc_cells.png'
    cell_fig.savefig(cell_path, dpi=FIGURE_DPI, bbox_inches='tight')
    print(f'Saved: {cell_path}')
plt.show()

In [ ]:
valid_cells = cell_qc[np.isfinite(cell_qc['zncc_residual']) & np.isfinite(cell_qc['center_pixel_residual'])]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].hexbin(
    valid_cells['center_pixel_residual'], valid_cells['zncc_residual'],
    gridsize=70, extent=(0, 1, 0, 1), mincnt=1, cmap='viridis',
)
axes[0].plot([0, 1], [0, 1], color='white', linestyle='--', linewidth=1)
axes[0].set_xlabel('Dense residual at nearest center pixel')
axes[0].set_ylabel('Neighborhood median assigned to cell')
axes[0].set_title('Aggregation consistency')
delta_values = valid_cells['aggregation_delta'].to_numpy(dtype=float)
delta_limit = max(float(np.percentile(np.abs(delta_values), 99)), 1e-6) if len(delta_values) else 1.0
axes[1].hist(delta_values, bins=100, range=(-delta_limit, delta_limit), color='slateblue', alpha=0.9)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Aggregated residual − center-pixel residual')
axes[1].set_ylabel('cells')
axes[1].set_title('Effect of cell-neighborhood median')
plt.show()

## Interpretation

- ZNCC correlation is local Pearson correlation over corresponding pixels: `1` is strong positive agreement, `0` is no linear pattern agreement, and negative values indicate inverse local patterns.
- ZNCC residual is `1 - clip(correlation, 0, 1)`: `0` is strong agreement and `1` is uncorrelated or negatively correlated.
- A cell receives the median dense value around its projected center. With the default settings at `2.6 µm/pixel`, the cell sampling neighborhood is 3×3, while the much larger ZNCC window supplies regional structural context.
- `NaN` means insufficient valid or variable signal, not failed alignment. Always inspect the valid fractions.
- Fixed correlation (`-1` to `1`) and residual (`0` to `1`) color limits make whole-slide, zoom, and cell plots directly comparable.
- This notebook evaluates alignment as stored. It does not estimate or apply a displacement correction.